In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import wandb

/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [34]:
DATA_PATH = Path("/home/pandeyps/Prefix/scM/architecture/subset.h5ad")
GROUP_COL = "sample"
LABEL_COL = "is_malignant"
SEED = 42
BATCH_SIZE = 1024
EPOCHS = 50
MUON_LR = 0.001
ADAMW_LR = 1e-4
WEIGHT_DECAY = 1e-2
MOMENTUM = 0.95
DROPOUT = 0.5
GRAD_CLIP = 1.0
CHECKPOINT_EVERY = 10
STAT_CHUNK_SIZE = 8192

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
# in backed mode so that adata loads on the disk
adata = sc.read_h5ad(DATA_PATH, backed="r")

In [6]:
adata.obs[LABEL_COL].value_counts()

is_malignant
0    281312
1    120084
Name: count, dtype: int64

In [7]:
# object with only sample and label columns
# typecast as string for sample and int for label
obs = adata.obs[[GROUP_COL, LABEL_COL]].copy()
obs[GROUP_COL] = obs[GROUP_COL].astype(str)
obs[LABEL_COL] = obs[LABEL_COL].astype(int)

# one label per sample by max label for each sample
group_labels = obs.groupby(GROUP_COL)[LABEL_COL].max()
groups = group_labels.index.to_numpy()
group_y = group_labels.to_numpy()

In [8]:
# split to train and temp (for stratification)
try:
    train_groups, temp_groups = train_test_split(
        groups,
        test_size=0.30,
        random_state=SEED,
        stratify=group_y,
    )
except ValueError:
    train_groups, temp_groups = train_test_split(
        groups,
        test_size=0.30,
        random_state=SEED,
    )

In [9]:
# take labels for temp samples
temp_y = group_labels.loc[temp_groups].to_numpy()

In [10]:
# validation and test split using temp
try:
    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_y,
    )
except ValueError:
    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=0.50,
        random_state=SEED,
    )

In [11]:
# sample to rows
train_idx = np.where(obs[GROUP_COL].isin(train_groups))[0]
val_idx = np.where(obs[GROUP_COL].isin(val_groups))[0]
test_idx = np.where(obs[GROUP_COL].isin(test_groups))[0]

In [12]:
print("train cells:", len(train_idx), "groups:", len(train_groups))
print("val cells:", len(val_idx), "groups:", len(val_groups))
print("test cells:", len(test_idx), "groups:", len(test_groups))
print("train labels:")
print(obs.iloc[train_idx][LABEL_COL].value_counts())
print("val labels:")
print(obs.iloc[val_idx][LABEL_COL].value_counts())
print("test labels:")
print(obs.iloc[test_idx][LABEL_COL].value_counts())

train cells: 296145 groups: 179
val cells: 47705 groups: 39
test cells: 57546 groups: 39
train labels:
is_malignant
0    199965
1     96180
Name: count, dtype: int64
val labels:
is_malignant
0    35594
1    12111
Name: count, dtype: int64
test labels:
is_malignant
0    45753
1    11793
Name: count, dtype: int64


In [13]:
# std and mean for genes 
def compute_gene_stats(indices, chunk_size=8192):
    indices = np.sort(np.asarray(indices))
    n = 0
    sums = np.zeros(adata.n_vars, dtype=np.float64)
    sq_sums = np.zeros(adata.n_vars, dtype=np.float64)
    for start in range(0, len(indices), chunk_size):
        batch_idx = indices[start:start + chunk_size]
        x = adata.X[batch_idx]
        if sp.issparse(x):
            sums += np.asarray(x.sum(axis=0)).ravel()
            sq_sums += np.asarray(x.multiply(x).sum(axis=0)).ravel()
        else:
            x = np.asarray(x, dtype=np.float64)
            sums += x.sum(axis=0)
            sq_sums += np.square(x).sum(axis=0)
        n += len(batch_idx)
    mean = sums / n
    var = np.maximum((sq_sums / n) - np.square(mean), 1e-6)
    std = np.sqrt(var)
    std[std < 1e-6] = 1.0

    return mean.astype(np.float32), std.astype(np.float32)


In [14]:
train_mean, train_std = compute_gene_stats(train_idx, STAT_CHUNK_SIZE)

In [15]:
print("mean range:", float(train_mean.min()), float(train_mean.max()))
print("std range:", float(train_std.min()), float(train_std.max()))

mean range: 0.10760245472192764 6.512973308563232
std range: 0.7117588520050049 3.8953514099121094


In [16]:
# dataset class to return indices for dataloader
class H5ADDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        return int(self.indices[i])

In [17]:
labels = obs[LABEL_COL].to_numpy().astype(np.float32)

In [18]:
# to sort indices and dense the sparse matrix for each batch
def collate_batch(batch_indices):
    batch_indices = np.asarray(batch_indices)
    order = np.argsort(batch_indices)
    sorted_idx = batch_indices[order]
    restore_order = np.argsort(order)

    x = adata.X[sorted_idx]

    if sp.issparse(x):
        x = x.toarray()

    x = np.asarray(x, dtype=np.float32)[restore_order]
    x = (x - train_mean) / train_std
    y = labels[sorted_idx][restore_order]

    return torch.from_numpy(x), torch.from_numpy(y).view(-1, 1)  # (x,y)

In [19]:
# shuffled
train_loader = DataLoader(
    H5ADDataset(train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)

In [20]:
# non shuffled
val_loader = DataLoader(
    H5ADDataset(val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

In [21]:
# non shuffled
test_loader = DataLoader(
    H5ADDataset(test_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

In [22]:
# used in muon optimizer for 3d grad updates
@torch.no_grad()
def zeropower_via_newtonschulz5(g, steps=5, eps=1e-7):
    original_dtype = g.dtype
    x = g.float()
    transposed = x.size(0) > x.size(1)

    if transposed:
        x = x.t()

    x = x / (x.norm() + eps)

    a, b, c = 3.4445, -4.7750, 2.0315
    for _ in range(steps):
        xx_t = x @ x.t()
        x = a * x + (b * xx_t + c * xx_t @ xx_t) @ x

    if transposed:
        x = x.t()

    return x.to(original_dtype)

In [23]:
# original reference used to write: 
# (decoupled weight decay)
class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.005, momentum=0.95, weight_decay=0.0, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            weight_decay = group["weight_decay"]
            ns_steps = group["ns_steps"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                grad = p.grad
                if grad.ndim != 2:
                    raise ValueError("Muon should only be used on 2D weight matrices.")

                if weight_decay != 0:
                    p.mul_(1.0 - lr * weight_decay)

                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p)

                buf = state["momentum_buffer"]
                buf.mul_(momentum).add_(grad)
                update = grad.add(buf, alpha=momentum)
                update = zeropower_via_newtonschulz5(update, steps=ns_steps)

                p.add_(update, alpha=-lr)

In [24]:
class scMFCA(nn.Module): # 128 -> 32 -> 1
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x.float())


In [25]:
model = scMFCA(adata.n_vars).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()

In [26]:
muon_params = []
adamw_params = []

In [27]:
# add params to the empty optimizer lists (based on shape)
for name, param in model.named_parameters():
    if param.ndim == 2 and min(param.shape) > 1:
        muon_params.append(param)
    else:
        adamw_params.append(param)

In [28]:
# param-ed objects
muon_optimizer = Muon(muon_params, lr=MUON_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
adamw_optimizer = torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=WEIGHT_DECAY)

In [29]:
print("Muon parameters:", sum(p.numel() for p in muon_params))
print("AdamW params:", sum(p.numel() for p in adamw_params))

Muon parameters: 482176
AdamW params: 513


In [30]:
model

scMFCA(
  (net): Sequential(
    (0): Linear(in_features=3735, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [31]:
wandb.init(
    project="scMFCA",
    name="scMFCA-run",
    config={
        "architecture": "CancerFinderMLP",
        "optimizer": "Muon + AdamW",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "muon_lr": MUON_LR,
        "adamw_lr": ADAMW_LR,
        "weight_decay": WEIGHT_DECAY,
        "momentum": MOMENTUM,
        "dropout": DROPOUT,
        "grad_clip": GRAD_CLIP,
        "input_dim": adata.n_vars,
        "train_cells": len(train_idx),
        "val_cells": len(val_idx),
        "test_cells": len(test_idx),
        "train_groups": len(train_groups),
        "val_groups": len(val_groups),
        "test_groups": len(test_groups),
    },
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/pandeyps/.netrc.
wandb: Currently logged in as: pandeyps (pandeyps-rictrlab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [32]:
wandb.watch(model)

In [35]:
def run_epoch(loader, training=False): #one pass
    model.train() if training else model.eval()

    total_loss = 0.0
    total_n = 0
    all_probs = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        with torch.set_grad_enabled(training):
            logits = model(x)
            loss_targets = y * 0.9 + 0.05 if training else y #smoothen
            loss = loss_fn(logits, loss_targets)

            if training:
                muon_optimizer.zero_grad(set_to_none=True)
                adamw_optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                muon_optimizer.step()
                adamw_optimizer.step()

        batch_size = y.size(0)
        total_loss += loss.item() * batch_size
        total_n += batch_size

        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        targets = y.detach().cpu().numpy().ravel()

        all_probs.extend(probs)
        all_targets.extend(targets)

    all_probs = np.asarray(all_probs)
    all_targets = np.asarray(all_targets).astype(int)
    preds = (all_probs >= 0.5).astype(int)

    metrics = {
        "loss": total_loss / total_n,
        "accuracy": accuracy_score(all_targets, preds),
        "f1": f1_score(all_targets, preds, zero_division=0),
        "auprc": average_precision_score(all_targets, all_probs),
        "auroc": np.nan,
    }

    if len(np.unique(all_targets)) == 2:
        metrics["auroc"] = roc_auc_score(all_targets, all_probs)

    return metrics


In [36]:
best_val_auprc = 0.0

In [37]:
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)

    wandb.log({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_f1": train_metrics["f1"],
        "train_auprc": train_metrics["auprc"],
        "train_auroc": train_metrics["auroc"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "val_auprc": val_metrics["auprc"],
        "val_auroc": val_metrics["auroc"],
    })

    print(
        f"epoch {epoch}/{EPOCHS} | "
        f"train loss: {train_metrics['loss']:.4f} | "
        f"val loss: {val_metrics['loss']:.4f} | "
        f"val AUPRC: {val_metrics['auprc']:.4f}"
    )

    if val_metrics["auprc"] > best_val_auprc:
        best_val_auprc = val_metrics["auprc"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "muon_optimizer_state_dict": muon_optimizer.state_dict(),
                "adamw_optimizer_state_dict": adamw_optimizer.state_dict(),
                "train_mean": train_mean,
                "train_std": train_std,
                "best_val_auprc": best_val_auprc,
                "train_metrics": train_metrics,
                "val_metrics": val_metrics,
            },
            "best-ckpt.pt",
        )
        wandb.save("best-ckpt.pt")

    if epoch % CHECKPOINT_EVERY == 0:
        checkpoint_path = f"checkpoint-epoch-{epoch}.pt"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "muon_optimizer_state_dict": muon_optimizer.state_dict(),
                "adamw_optimizer_state_dict": adamw_optimizer.state_dict(),
                "train_mean": train_mean,
                "train_std": train_std,
                "best_val_auprc": best_val_auprc,
                "train_metrics": train_metrics,
                "val_metrics": val_metrics,
            },
            checkpoint_path,
        )
        wandb.save(checkpoint_path)

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch 1/50 | train loss: 0.6014 | val loss: 0.4802 | val AUPRC: 0.7482
epoch 2/50 | train loss: 0.4230 | val loss: 0.4046 | val AUPRC: 0.7918
epoch 3/50 | train loss: 0.3395 | val loss: 0.3579 | val AUPRC: 0.8164
epoch 4/50 | train loss: 0.3028 | val loss: 0.3285 | val AUPRC: 0.8329
epoch 5/50 | train loss: 0.2835 | val loss: 0.3143 | val AUPRC: 0.8401
epoch 6/50 | train loss: 0.2715 | val loss: 0.3000 | val AUPRC: 0.8480
epoch 7/50 | train loss: 0.2629 | val loss: 0.2934 | val AUPRC: 0.8488
epoch 8/50 | train loss: 0.2570 | val loss: 0.2898 | val AUPRC: 0.8504
epoch 9/50 | train loss: 0.2526 | val loss: 0.2817 | val AUPRC: 0.8548


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch 10/50 | train loss: 0.2490 | val loss: 0.2793 | val AUPRC: 0.8556
epoch 11/50 | train loss: 0.2464 | val loss: 0.2804 | val AUPRC: 0.8523
epoch 12/50 | train loss: 0.2441 | val loss: 0.2771 | val AUPRC: 0.8549
epoch 13/50 | train loss: 0.2420 | val loss: 0.2763 | val AUPRC: 0.8551
epoch 14/50 | train loss: 0.2402 | val loss: 0.2730 | val AUPRC: 0.8582
epoch 15/50 | train loss: 0.2390 | val loss: 0.2740 | val AUPRC: 0.8552
epoch 16/50 | train loss: 0.2371 | val loss: 0.2730 | val AUPRC: 0.8560
epoch 17/50 | train loss: 0.2361 | val loss: 0.2732 | val AUPRC: 0.8559
epoch 18/50 | train loss: 0.2350 | val loss: 0.2709 | val AUPRC: 0.8569
epoch 19/50 | train loss: 0.2336 | val loss: 0.2716 | val AUPRC: 0.8568


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


...

epoch 30/50 | train loss: 0.2252 | val loss: 0.2674 | val AUPRC: 0.8609
epoch 31/50 | train loss: 0.2248 | val loss: 0.2675 | val AUPRC: 0.8593
epoch 32/50 | train loss: 0.2242 | val loss: 0.2674 | val AUPRC: 0.8618
epoch 33/50 | train loss: 0.2236 | val loss: 0.2691 | val AUPRC: 0.8579
epoch 34/50 | train loss: 0.2236 | val loss: 0.2687 | val AUPRC: 0.8588
epoch 35/50 | train loss: 0.2230 | val loss: 0.2681 | val AUPRC: 0.8578
epoch 36/50 | train loss: 0.2227 | val loss: 0.2688 | val AUPRC: 0.8584
epoch 37/50 | train loss: 0.2222 | val loss: 0.2651 | val AUPRC: 0.8626
epoch 38/50 | train loss: 0.2219 | val loss: 0.2687 | val AUPRC: 0.8578
epoch 39/50 | train loss: 0.2216 | val loss: 0.2666 | val AUPRC: 0.8596


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch 40/50 | train loss: 0.2212 | val loss: 0.2664 | val AUPRC: 0.8615
epoch 41/50 | train loss: 0.2211 | val loss: 0.2672 | val AUPRC: 0.8605
epoch 42/50 | train loss: 0.2209 | val loss: 0.2661 | val AUPRC: 0.8605
epoch 43/50 | train loss: 0.2209 | val loss: 0.2647 | val AUPRC: 0.8633
epoch 44/50 | train loss: 0.2204 | val loss: 0.2664 | val AUPRC: 0.8608
epoch 45/50 | train loss: 0.2201 | val loss: 0.2649 | val AUPRC: 0.8618
epoch 46/50 | train loss: 0.2198 | val loss: 0.2654 | val AUPRC: 0.8614
epoch 47/50 | train loss: 0.2198 | val loss: 0.2657 | val AUPRC: 0.8608
epoch 48/50 | train loss: 0.2194 | val loss: 0.2686 | val AUPRC: 0.8592
epoch 49/50 | train loss: 0.2195 | val loss: 0.2666 | val AUPRC: 0.8612


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch 50/50 | train loss: 0.2192 | val loss: 0.2691 | val AUPRC: 0.8575


In [38]:
checkpoint = torch.load("best-ckpt.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [39]:
# get logits to temperature scale 
def logits(loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x)
            logits_list.append(logits.detach())
            targets_list.append(y.to(DEVICE))

    return torch.cat(logits_list), torch.cat(targets_list)

In [40]:
def metrics_from_logits(logits, targets):
    loss = F.binary_cross_entropy_with_logits(logits, targets).item()
    probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
    y_true = targets.detach().cpu().numpy().ravel().astype(int)
    preds = (probs >= 0.5).astype(int)

    metrics = {
        "loss": loss,
        "accuracy": accuracy_score(y_true, preds),
        "f1": f1_score(y_true, preds, zero_division=0),
        "auprc": average_precision_score(y_true, probs),
        "auroc": np.nan,
    }

    if len(np.unique(y_true)) == 2:
        metrics["auroc"] = roc_auc_score(y_true, probs)

    return metrics

In [41]:
val_logits, val_targets = logits(val_loader)
test_logits, test_targets = logits(test_loader)

In [42]:
log_temperature = torch.zeros(1, device=DEVICE, requires_grad=True)
temperature_optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=50)

In [43]:
# minimize val BCE [1]
def temperature_closure():
    temperature_optimizer.zero_grad()
    temperature = log_temperature.exp()
    loss = F.binary_cross_entropy_with_logits(val_logits / temperature, val_targets)
    loss.backward()
    return loss

In [44]:
temperature_optimizer.step(temperature_closure)
temperature = log_temperature.exp().detach()

In [45]:
val_metrics = metrics_from_logits(val_logits, val_targets)
calibrated_val_metrics = metrics_from_logits(val_logits / temperature, val_targets) # [1]
test_metrics = metrics_from_logits(test_logits, test_targets)
calibrated_test_metrics = metrics_from_logits(test_logits / temperature, test_targets) # [1]

In [46]:
wandb.log({
    "temperature": float(temperature.item()),
    "val_loss_uncalibrated": val_metrics["loss"],
    "val_loss_calibrated": calibrated_val_metrics["loss"],
    "test_loss_uncalibrated": test_metrics["loss"],
    "test_loss_calibrated": calibrated_test_metrics["loss"],
    "test_accuracy_calibrated": calibrated_test_metrics["accuracy"],
    "test_f1_calibrated": calibrated_test_metrics["f1"],
    "test_auprc_calibrated": calibrated_test_metrics["auprc"],
    "test_auroc_calibrated": calibrated_test_metrics["auroc"],
})

In [47]:
print("temperature:", float(temperature.item()))
print("validation before calibration:", val_metrics)
print("validation after calibration:", calibrated_val_metrics)
print("test before calibration:", test_metrics)
print("test after calibration:", calibrated_test_metrics)

temperature: 1.0041922330856323
validation before calibration: {'loss': 0.2646586000919342, 'accuracy': 0.9098207735038256, 'f1': 0.8081006334195736, 'auprc': 0.8632737355902671, 'auroc': 0.9083754125178383}
validation after calibration: {'loss': 0.264654278755188, 'accuracy': 0.9098207735038256, 'f1': 0.8081006334195736, 'auprc': 0.8632737731721368, 'auroc': 0.9083754136777187}
test before calibration: {'loss': 0.11635686457157135, 'accuracy': 0.9722482883258611, 'f1': 0.9334333708474011, 'auprc': 0.9607543085047524, 'auroc': 0.9913784717544265}
test after calibration: {'loss': 0.11685645580291748, 'accuracy': 0.9722482883258611, 'f1': 0.9334333708474011, 'auprc': 0.9607543122738877, 'auroc': 0.9913784717544266}


In [48]:
calibrated_test_metrics

{'loss': 0.11685645580291748,
 'accuracy': 0.9722482883258611,
 'f1': 0.9334333708474011,
 'auprc': 0.9607543122738877,
 'auroc': 0.9913784717544266}

In [49]:
model.eval()
test_probs = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x) / temperature
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        test_probs.extend(probs)


In [50]:
predictions = pd.DataFrame({
    "cell_id": adata.obs_names[test_idx],
    "sample": obs.iloc[test_idx][GROUP_COL].values,
    "true_label": labels[test_idx].astype(int),
    "predicted_probability": test_probs,
})

In [51]:
predictions["predicted_label"] = (predictions["predicted_probability"] >= 0.5).astype(int) # .5 as default threshold

In [52]:
wandb.save("best-ckpt.pt")

['/home/pandeyps/Prefix/scM/architecture/wandb/run-20260622_193813-tv62dimk/files/best-ckpt.pt']

In [53]:
predictions.head(100).to_string()

'                                       cell_id  sample  true_label  predicted_probability  predicted_label\n0   BCH836-P01-B12-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.039340                0\n1   BCH836-P01-C06-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.036533                0\n2   BCH836-P01-C09-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.037827                0\n3   BCH836-P01-C10-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.035754                0\n4   BCH836-P01-C12-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.043472                0\n5   BCH836-P01-D08-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           1               0.944924                1\n6   BCH836-P01-D12-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.076073                0\n7   BCH836-P01-G01-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.038607                0\n8   BCH836-P01-G11-3-0-0-0-

In [55]:
# wandb.finish()
# adata.file.close()

                                       cell_id  sample  true_label  predicted_probability  predicted_label
0   BCH836-P01-B12-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.035891                0
1   BCH836-P01-C06-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.031269                0
2   BCH836-P01-C09-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.038245                0
3   BCH836-P01-C10-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.036813                0
4   BCH836-P01-C12-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.030757                0
5   BCH836-P01-D08-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           1               0.925250                1
6   BCH836-P01-D12-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.077936                0
7   BCH836-P01-G01-3-0-0-0-0-0-0-0-0-0-0-0-0-0  BCH836           0               0.024148                0
8   BCH836-P01-G11-3-0-0-0-0-0-0-0-0-